<a href="https://colab.research.google.com/github/mjcowley/Python_Intro/blob/main/02-NumPy-and-Astropy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/mjcowley/Python_Intro/main/Images/qarg_bw_dark.png" alt="QUT Astrophysics Research Group" width="400"/>

# Part 2: NumPy and Astropy

**QUT Astrophysics Research Group (QARG) Python Training**

Group leads: Dr Michael Cowley and Dr Rossana Ruggeri
Author: Michael Cowley
Last updated: 18 Oct 2026

## Overview

In Part 1 you worked with one galaxy at a time, using loops when you had several. Real catalogues have thousands or millions of objects, so we need tools that work on whole sets of numbers at once. That's what **NumPy** does, and almost every scientific Python library is built on it.

You'll also meet **Astropy**, the community library for astronomy. It handles physical units and constants, so you don't have to keep track of factors of $10^{-7}$ by hand, and it does cosmological calculations like luminosity distances and lookback times.

This notebook takes about **2.5 hours**. It doesn't depend on Part 1 having been run, but it assumes you're comfortable with what Part 1 covered.

### By the end of Part 2 you'll be able to

- Create NumPy arrays and do maths on them without loops
- Select the objects you want using boolean masks, and deal with missing values
- Work with 2D arrays, including simple aperture photometry on an image
- Attach units to numbers and convert between them with `astropy.units`
- Use physical constants from `astropy.constants`
- Calculate distances, ages and physical sizes at any redshift with `astropy.cosmology`

## Setup

Run this cell first. Colab already has everything installed. If you're running locally and Astropy is missing, the cell installs it for you.

In [ ]:
try:
    import astropy
except ImportError:
    %pip install astropy
    import astropy

import math
import numpy as np
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)
print("Astropy version:", astropy.__version__)

---
## 1. Why NumPy?

Say you have the $K_s$ fluxes of some galaxies, in µJy, and want their AB magnitudes. With a plain Python list you'd need a loop:

In [ ]:
fluxes = [45.2, 12.8, 8.6, 60.1, 5.3]

mags = []
for f in fluxes:
    mags.append(-2.5 * math.log10(f) + 23.9)
print(mags)

With a NumPy **array**, you write the maths once and it applies to every element:

In [ ]:
fluxes = np.array([45.2, 12.8, 8.6, 60.1, 5.3])

mags = -2.5 * np.log10(fluxes) + 23.9
print(mags)

This is called **vectorisation**. It's shorter, it reads like the equation, and it's faster, because the loop happens inside NumPy's compiled code instead of in Python. Let's time both for a million galaxies:

In [ ]:
import time

big_list = list(range(1, 1_000_001))
big_array = np.arange(1, 1_000_001)

start = time.perf_counter()
result = [-2.5 * math.log10(f) + 23.9 for f in big_list]
t_list = time.perf_counter() - start

start = time.perf_counter()
result = -2.5 * np.log10(big_array) + 23.9
t_array = time.perf_counter() - start

print(f"List:  {t_list * 1000:.1f} ms")
print(f"Array: {t_array * 1000:.1f} ms  ({t_list / t_array:.0f} times faster)")

### Arrays aren't lists

Arrays look like lists, but maths on them behaves differently. This is a common source of bugs:

In [ ]:
my_list = [1, 2, 3]
my_array = np.array([1, 2, 3])

print(my_list * 2)       # repeats the list
print(my_array * 2)      # multiplies each element

print(my_list + [10])    # joins two lists
print(my_array + 10)     # adds 10 to each element

Also, every element of an array has the same type. If you mix integers and floats, NumPy converts everything to floats.

---
## 2. Creating Arrays

You'll rarely type arrays out by hand. NumPy has functions to build them:

| Function | Gives you |
|----------|-----------|
| `np.array(a_list)` | An array from a list |
| `np.arange(start, stop, step)` | Evenly spaced values with a given step (like `range`, `stop` not included) |
| `np.linspace(start, stop, n)` | `n` evenly spaced values, with both ends included |
| `np.logspace(a, b, n)` | `n` values spaced evenly in log, from $10^a$ to $10^b$ |
| `np.zeros(n)`, `np.ones(n)` | `n` zeros or ones |

In [ ]:
print(np.arange(0, 1, 0.2))
print(np.linspace(0, 1, 6))
print(np.logspace(0, 3, 4))      # 10^0 to 10^3
print(np.zeros(3))

`np.logspace` is very useful in astronomy, where quantities often span many orders of magnitude. A wavelength grid from the UV to the far-infrared, for example, is much better sampled in log.

Every array knows its own size and shape:

In [ ]:
wavelengths = np.linspace(100, 1000, 10)   # nm

print(wavelengths)
print("Size: ", wavelengths.size)
print("Shape:", wavelengths.shape)
print("Type: ", wavelengths.dtype)

---
## 3. Vectorised Maths

Arithmetic between an array and a single number applies to every element. Arithmetic between two arrays of the same length works element by element.

In [ ]:
z = np.array([0.5, 1.0, 2.0, 3.0])

scale_factor = 1 / (1 + z)
print(scale_factor)

# Observed wavelength of the H-alpha line (656.3 nm in the rest frame) at each redshift
print(656.3 * (1 + z))

In [ ]:
u_mag = np.array([22.1, 23.4, 21.8])
v_mag = np.array([21.0, 22.9, 20.1])

print(u_mag - v_mag)        # U-V colour of each galaxy

NumPy has vectorised versions of all the usual mathematical functions: `np.sqrt`, `np.exp`, `np.log` (natural), `np.log10`, `np.sin`, `np.cos`, `np.radians` and many more. Use these with arrays, not the `math` versions, which only work on single numbers.

Functions you write yourself work on arrays too, as long as they use NumPy functions inside:

In [ ]:
def ab_mag(flux_ujy):
    """AB magnitude for a flux density in microjanskys."""
    return -2.5 * np.log10(flux_ujy) + 23.9


print(ab_mag(45.2))                           # a single number still works
print(ab_mag(np.array([1.0, 10.0, 100.0])))   # and so does an array

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Fluxes and Magnitudes
  </h1>
</div>

1. Write a function `ab_flux(mag)` that does the reverse of `ab_mag()`, returning the flux in µJy for an AB magnitude. Rearranging the formula gives $f = 10^{(23.9 - m)/2.5}$. In Python, powers use `**`.
2. ZFOURGE reaches galaxies as faint as about $K_s = 25$. Make an array called `mag_grid` of magnitudes from 18 to 25 in steps of 0.5, and use your function to make a matching array of fluxes called `flux_grid`.
3. Check your function by converting `flux_grid` back to magnitudes with `ab_mag()`. You should get `mag_grid` back.
4. How many times fainter is a $K_s = 25$ galaxy than a $K_s = 18$ one?

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert abs(ab_flux(23.9) - 1.0) < 1e-9, "ab_flux(23.9) should be 1 µJy. Check your formula."
assert len(mag_grid) == 15 and mag_grid[-1] == 25, "mag_grid should run from 18 to 25 inclusive, in steps of 0.5."
assert np.allclose(ab_mag(flux_grid), mag_grid), "Converting flux_grid back to magnitudes should give mag_grid."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
def ab_flux(mag):
    """Flux density in microjanskys for an AB magnitude."""
    return 10 ** ((23.9 - mag) / 2.5)


mag_grid = np.arange(18, 25.5, 0.5)     # stop at 25.5 so that 25 is included
flux_grid = ab_flux(mag_grid)
print(flux_grid)

print(ab_mag(flux_grid))                # should match mag_grid

print(f"A Ks = 25 galaxy is {flux_grid[0] / flux_grid[-1]:.0f} times fainter than a Ks = 18 one")
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

Every 5 magnitudes is exactly a factor of 100 in flux, so 7 magnitudes is $100^{7/5} \approx 631$.

Notice the check above uses `np.allclose()`. It's the array version of the "are these floats close enough?" test from Part 1.

---
## 4. Indexing, Masks and Missing Data

### Indexing and slicing

Arrays are indexed and sliced exactly like lists. Unlike lists, you can also pick out several elements at once by giving a list of indices.

In [ ]:
z = np.array([0.62, 0.91, 1.34, 0.75, 1.12, 0.58, 1.48, 0.83])

print(z[0], z[-1])
print(z[2:5])
print(z[[0, 3, 6]])      # elements 0, 3 and 6

### Boolean masks

This is one of the most useful ideas in NumPy. A comparison on an array gives an array of `True` and `False` values, called a **mask**. Using a mask as an index keeps only the elements where it's `True`.

In [ ]:
high_z = z > 1
print(high_z)
print(z[high_z])

Since `True` counts as 1 and `False` as 0, summing a mask counts how many elements pass:

In [ ]:
print(f"{np.sum(high_z)} of {z.size} galaxies are above z = 1")
print(f"That's {np.mean(high_z):.0%} of the sample")

Masks from one array can be used to select from another array of the same length. This is how you'll select samples from catalogues.

In [ ]:
ks_mag = np.array([19.76, 21.13, 21.56, 19.45, 22.09, 20.52, 20.91, 21.41])

print("Magnitudes of the z > 1 galaxies:", ks_mag[z > 1])

### Combining conditions

To combine masks, use `&` (and), `|` (or) and `~` (not), **with brackets around each condition**. The words `and` and `or` from Part 1 don't work on arrays.

In [ ]:
sample = (z > 0.6) & (z < 1.2) & (ks_mag < 21.5)
print(sample)
print("Selected redshifts:", z[sample])

In [ ]:
# Leaving out the brackets gives an error, because & is applied before > and <
sample = z > 0.6 & z < 1.2

### Missing data

Catalogues often mark missing measurements with a special **sentinel value**, such as `-99`, instead of leaving them blank. If you don't catch these, they'll quietly ruin your averages.

In [ ]:
flux = np.array([45.2, 12.8, -99.0, 60.1, 5.3, -99.0])

print("Wrong mean:", np.mean(flux))

NumPy's way to mark a missing value is `np.nan` ("not a number"). Any maths involving `nan` gives `nan`, which is annoying but safe, since you can't accidentally get a wrong number. Functions like `np.nanmean` skip them, and `np.isfinite` makes a mask of the good values.

In [ ]:
flux = np.where(flux == -99, np.nan, flux)    # where flux is -99 use nan, otherwise keep flux
print(flux)

print("np.mean:   ", np.mean(flux))
print("np.nanmean:", np.nanmean(flux))
print("Good values:", flux[np.isfinite(flux)])

`np.where(condition, a, b)` is worth remembering in general. It picks from `a` where the condition is true and from `b` where it's false.

### A warning about copies

Assigning an array to a new name doesn't copy it. Both names point to the same data, so changing one changes the other. Use `.copy()` when you want an independent copy.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = a              # b is the same array as a
c = a.copy()       # c is a new, independent array

b[0] = 100
print("a:", a)     # a has changed too!
print("c:", c)

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Selecting a Sample
  </h1>
</div>

Here are photometric redshifts and $K_s$ magnitudes for 20 galaxies. Missing magnitudes are marked with `-99`.

```python
z_phot = np.array([0.45, 1.21, 2.34, 0.87, 1.56, 3.12, 0.33, 1.89, 2.71, 1.05,
                   0.92, 1.44, 2.05, 0.61, 1.73, 2.88, 1.17, 0.78, 1.98, 1.36])
ks = np.array([20.1, 22.4, 23.8, 21.0, -99, 24.6, 19.5, 22.9, 24.1, 21.7,
               21.2, 22.0, -99, 20.4, 22.6, 24.9, 21.9, 20.8, 23.3, 22.2])
```

1. Replace the `-99` values in `ks` with `np.nan`, and print the mean $K_s$ magnitude of the galaxies that have one.
2. Make a mask called `sample` that selects galaxies with $1 < z < 2$ **and** $K_s < 23$. Galaxies with missing magnitudes shouldn't be selected. (Check what `np.nan < 23` gives you.)
3. Print how many galaxies are in your sample, and their redshifts.

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert np.sum(np.isnan(ks)) == 2, "ks should have exactly two nan values."
assert np.sum(sample) == 7, "Your sample should have 7 galaxies. Check each of the three conditions."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
z_phot = np.array([0.45, 1.21, 2.34, 0.87, 1.56, 3.12, 0.33, 1.89, 2.71, 1.05,
                   0.92, 1.44, 2.05, 0.61, 1.73, 2.88, 1.17, 0.78, 1.98, 1.36])
ks = np.array([20.1, 22.4, 23.8, 21.0, -99, 24.6, 19.5, 22.9, 24.1, 21.7,
               21.2, 22.0, -99, 20.4, 22.6, 24.9, 21.9, 20.8, 23.3, 22.2])

ks = np.where(ks == -99, np.nan, ks)
print(f"Mean Ks: {np.nanmean(ks):.2f}")

# Comparisons with nan are always False, so missing magnitudes drop out automatically
sample = (z_phot > 1) & (z_phot < 2) & (ks < 23)
print(f"{np.sum(sample)} galaxies selected: {z_phot[sample]}")
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

Any comparison with `nan` gives `False`, which is why galaxy 5 (at $z = 1.56$, with no magnitude) dropped out of the sample on its own. That's usually what you want, but it's worth checking every time.

---
## 5. Statistics and Random Numbers

### Summary statistics

NumPy has functions for all the usual summary statistics. Each of these works on a whole array:

In [ ]:
z = np.array([0.62, 0.91, 1.34, 0.75, 1.12, 0.58, 1.48, 0.83])

print("Mean:      ", np.mean(z))
print("Median:    ", np.median(z))
print("Std dev:   ", np.std(z, ddof=1))       # ddof=1 gives the sample standard deviation
print("Min, max:  ", np.min(z), np.max(z))
print("16th, 84th percentiles:", np.percentile(z, [16, 84]))
print("Index of the highest-z galaxy:", np.argmax(z))

The 16th and 84th percentiles are common in astronomy. For a normal distribution they sit one standard deviation either side of the mean, and they still mean something when the distribution isn't normal. You'll see them as error bars on photometric redshifts and masses from SED fitting. Part 5 covers statistics in more depth.

### Random numbers

Simulating data is a great way to test your analysis, since you know what the right answer is. NumPy's random number generator does this. Giving it a **seed** means you get the same "random" numbers every time, so your results are reproducible.

In [ ]:
rng = np.random.default_rng(seed=42)

print(rng.normal(loc=0, scale=1, size=5))       # normal (Gaussian) distribution
print(rng.uniform(low=0, high=10, size=5))      # uniform between 0 and 10
print(rng.poisson(lam=3, size=5))               # Poisson, for counting photons or galaxies

Here's an example. Suppose we measure a galaxy's flux 25 times and each measurement has random noise of 5 µJy. How well do we know the true flux?

In [ ]:
true_flux = 50.0
noise = 5.0

measurements = rng.normal(true_flux, noise, size=25)

mean = np.mean(measurements)
std = np.std(measurements, ddof=1)
error_on_mean = std / np.sqrt(measurements.size)

print(f"Mean of the measurements: {mean:.2f} µJy")
print(f"Scatter of the measurements: {std:.2f} µJy")
print(f"Uncertainty on the mean: {error_on_mean:.2f} µJy")

The scatter of individual measurements stays at about 5 µJy however many you take, but the uncertainty on the mean shrinks as $1/\sqrt{N}$. This is why astronomers take long exposures, or stack many frames together.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: The Power of Stacking
  </h1>
</div>

A faint galaxy has a true flux of 2 µJy, but each image only reaches a noise level of 5 µJy, so it can't be seen in any single image.

1. Using `rng.normal()`, simulate measuring the galaxy in $N = 10$ images. Calculate the mean flux and its uncertainty, and the **signal-to-noise ratio** (S/N), which is the mean divided by its uncertainty.
2. Repeat for $N = 100$ and $N = 1000$. Use a loop over the three values of $N$.
3. Astronomers usually call something a detection when S/N > 5. Roughly how many images would you need to stack to detect this galaxy? Work it out from the $1/\sqrt{N}$ scaling, and store your answer in `n_needed`.

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert 140 <= n_needed <= 170, "Not quite. You need true_flux / (noise / sqrt(N)) = 5. Solve for N."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
true_flux = 2.0
noise = 5.0

for n in [10, 100, 1000]:
    m = rng.normal(true_flux, noise, size=n)
    mean = np.mean(m)
    err = np.std(m, ddof=1) / np.sqrt(n)
    print(f"N = {n:4d}: flux = {mean:5.2f} ± {err:.2f} µJy, S/N = {mean / err:.1f}")

# We need true_flux / (noise / sqrt(N)) = 5, so sqrt(N) = 5 * noise / true_flux
n_needed = (5 * noise / true_flux) ** 2
print(f"About {n_needed:.0f} images needed for S/N = 5")
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

---
## 6. Two-Dimensional Arrays

An astronomical image is just a 2D array of numbers, one per pixel. NumPy handles arrays with any number of dimensions in the same way.

In [ ]:
grid = np.array([[1, 2, 3],
                 [4, 5, 6]])

print(grid.shape)       # (rows, columns)
print(grid[0, 2])       # row 0, column 2
print(grid[1, :])       # all of row 1
print(grid[:, 0])       # all of column 0

For images, **the first index is the row (y) and the second is the column (x)**, so a pixel is `image[y, x]`. This catches out almost everyone at some point.

Summary statistics can work on the whole array or along one direction, using the `axis` argument:

In [ ]:
print(np.sum(grid))             # everything
print(np.sum(grid, axis=0))     # add up each column
print(np.sum(grid, axis=1))     # add up each row

### Simulating a galaxy image

Let's make a fake image of a small, faint galaxy: a 2D Gaussian blob on a flat background, plus noise. `np.meshgrid` gives the x and y coordinate of every pixel, which lets us write the Gaussian as a single vectorised expression.

In [ ]:
rng = np.random.default_rng(seed=1)

ny, nx = 60, 60
y, x = np.mgrid[0:ny, 0:nx]         # y and x coordinate of every pixel (a shortcut for np.meshgrid)

x0, y0 = 32.0, 27.0                 # centre of the galaxy
sigma = 2.5                         # size of the galaxy, in pixels
total_flux = 2000.0                 # total counts from the galaxy
background = 10.0                   # counts per pixel from the sky

r2 = (x - x0) ** 2 + (y - y0) ** 2
galaxy = total_flux / (2 * np.pi * sigma ** 2) * np.exp(-r2 / (2 * sigma ** 2))

image = galaxy + background + rng.normal(0, 3.0, size=(ny, nx))

print(image.shape)

We'll cover plotting properly in Part 4, but here's a quick look at the image:

In [ ]:
plt.imshow(image, origin="lower", cmap="magma")
plt.colorbar(label="Counts")
plt.xlabel("x (pixels)")
plt.ylabel("y (pixels)")
plt.show()

`origin="lower"` puts pixel (0, 0) at the bottom left, which is the convention for astronomical images.

### Aperture photometry

To measure the galaxy's brightness, we add up all the pixels within some radius of its centre (an **aperture**). We can make a 2D mask for this just like the 1D masks earlier:

In [ ]:
r = np.sqrt((x - x0) ** 2 + (y - y0) ** 2)    # distance of every pixel from the centre

aperture = r < 8
print(f"{np.sum(aperture)} pixels in the aperture")
print(f"Sum inside the aperture: {np.sum(image[aperture]):.0f} counts")

That's far more than the 2000 counts we put in, because the sum also includes the sky background in every pixel. We need to subtract it.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Background Subtraction
  </h1>
</div>

To estimate the sky background, astronomers use an **annulus**: a ring around the object, far enough out to contain no light from it.

1. Make a mask called `annulus` that selects pixels with $15 \le r < 20$.
2. Estimate the background per pixel as the **median** of the image in the annulus, and store it in `bkg_per_pixel`. (The median is less affected than the mean by any stray stars in the annulus.)
3. Subtract the background from every pixel in the aperture, and add up what's left to get the galaxy's flux. Store it in `galaxy_flux`.
4. Compare with the true total flux of 2000 counts. Why might it be a little lower? *Hint: think about how far a Gaussian extends.*

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert abs(bkg_per_pixel - 10) < 0.5, "The background should be close to 10 counts per pixel."
assert abs(galaxy_flux - 2000) < 100, "Your galaxy flux should be close to 2000. Did you subtract the background from every aperture pixel?"
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
annulus = (r >= 15) & (r < 20)
bkg_per_pixel = np.median(image[annulus])

n_pix = np.sum(aperture)
galaxy_flux = np.sum(image[aperture]) - bkg_per_pixel * n_pix

print(f"Background: {bkg_per_pixel:.2f} counts per pixel")
print(f"Galaxy flux: {galaxy_flux:.0f} counts (true value: {total_flux:.0f})")

# The aperture only captures 1 - exp(-r^2 / 2 sigma^2) of a Gaussian's light, about 99.4% for r = 8
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

---
## 7. Units with Astropy

Astronomy uses a bewildering mix of units: parsecs and light years, ergs and joules, janskys, solar masses, angstroms. Unit mistakes are one of the most common sources of wrong answers, and occasionally of lost spacecraft.

`astropy.units` lets you attach units to numbers. Astropy then keeps track of them through your calculations, converts between them, and refuses to do calculations that don't make physical sense.

In [ ]:
import astropy.units as u

distance = 16.4 * u.Mpc
print(distance)
print(f"{distance.to(u.lyr):.3e}")
print(distance.to(u.km))

A number with units attached is called a **Quantity**. `.value` gives you the bare number back, and `.unit` the unit. Units combine automatically when you multiply and divide:

In [ ]:
speed = 1500 * u.km / u.s
time = 1 * u.Gyr

travelled = speed * time
print(travelled)                # astropy keeps the units as given
print(travelled.to(u.kpc))      # until you ask it to convert

print(travelled.to(u.kpc).value)

Astropy stops you doing something that makes no sense:

In [ ]:
distance.to(u.s)

### Equivalencies

Some conversions only make sense in a particular physical context. Wavelength and frequency are different kinds of quantity, but light of a given wavelength has a definite frequency. Astropy calls these **equivalencies**:

In [ ]:
ks_wavelength = 2.15 * u.micron

print(ks_wavelength.to(u.GHz, equivalencies=u.spectral()))
print(ks_wavelength.to(u.eV, equivalencies=u.spectral()))    # photon energy

Flux densities work the same way. Converting between $f_\nu$ (per unit frequency, like janskys) and $f_\lambda$ (per unit wavelength) depends on the wavelength, so you pass it in:

In [ ]:
f_nu = 45.2 * u.uJy
f_lambda = f_nu.to(u.erg / u.s / u.cm**2 / u.AA, equivalencies=u.spectral_density(ks_wavelength))
print(f_lambda)

### Physical constants

`astropy.constants` has all the physical and astronomical constants you're likely to need, with their units and references:

In [ ]:
from astropy import constants as const

print(const.c)
print()
print(const.G.to(u.pc * u.km**2 / u.s**2 / u.Msun))     # G in units handy for galaxy dynamics
print()
print(f"Speed of light: {const.c.to(u.km / u.s):.0f}")
print(f"Solar mass:     {const.M_sun:.3e}")
print(f"Solar luminosity: {const.L_sun:.3e}")

For example, the circular velocity of a star orbiting at radius $r$ around a mass $M$ is $v = \sqrt{GM/r}$. Let's find the Sun's orbital speed in the Milky Way, which has about $10^{11}$ solar masses inside the Sun's orbit at 8 kpc:

In [ ]:
M = 1e11 * u.Msun
r = 8 * u.kpc

v = np.sqrt(const.G * M / r)
print(v.to(u.km / u.s))

That's close to the measured value of about 230 km/s. Notice that we never had to look up $G$ or convert kiloparsecs to metres. Astropy did it all.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Blackbodies
  </h1>
</div>

Stars are approximately **blackbodies**, which emit light with a spectrum that depends only on their temperature $T$. The spectrum per unit wavelength is given by the Planck function:

$$B_\lambda(T) = \frac{2hc^2}{\lambda^5} \frac{1}{\exp\left(\dfrac{hc}{\lambda k_B T}\right) - 1}$$

1. Write a function `planck(wavelength, T)` that returns $B_\lambda$, using `const.h`, `const.c` and `const.k_B`. Convert the result to `u.W / u.m**2 / u.nm` before returning it. (Strictly it's also per steradian, which we'll leave off here.)
2. Make a wavelength grid with `np.linspace(100, 3000, 2901) * u.nm`, and calculate $B_\lambda$ for the Sun ($T = 5772$ K).
3. Use `np.argmax` to find the wavelength where the spectrum peaks, and store it in `peak_sun`.
4. Repeat for a hot O-type star ($T = 40{,}000$ K) and a cool M dwarf ($T = 3000$ K). **Wien's law** says the peak wavelength is $\lambda_\mathrm{peak} = b / T$ with $b = 2.898 \times 10^{-3}$ m K. Do your results agree?

You can explore blackbody curves interactively with the [Blackbody demo](https://mjcowley.github.io/astro_demos/blackbody/).

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert abs(peak_sun.to(u.nm).value - 502) < 3, "The Sun's spectrum should peak at about 502 nm."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
def planck(wavelength, T):
    """Blackbody spectral radiance per unit wavelength (per steradian omitted)."""
    x = const.h * const.c / (wavelength * const.k_B * T)
    b = 2 * const.h * const.c**2 / wavelength**5 / (np.exp(x) - 1)
    return b.to(u.W / u.m**2 / u.nm)


wavelength = np.linspace(100, 3000, 2901) * u.nm

for name, T in [("Sun", 5772 * u.K), ("O star", 40000 * u.K), ("M dwarf", 3000 * u.K)]:
    spectrum = planck(wavelength, T)
    peak = wavelength[np.argmax(spectrum)]
    wien = (2.898e-3 * u.m * u.K / T).to(u.nm)
    print(f"{name:8s}: peak at {peak:.0f}, Wien's law gives {wien:.0f}")
    if name == "Sun":
        peak_sun = peak
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

The O star's peak isn't quite right. Wien's law puts it at 72 nm, but our grid starts at 100 nm, so `np.argmax` just finds the edge of the grid. It's a good reminder to always check that your answer makes sense, and to plot things when you can.

---
## 8. Cosmology with Astropy

In Part 1 we used Hubble's law, $d = cz / H_0$, and noted that it only works for nearby galaxies. For the distant galaxies we study, the distance depends on how the Universe has expanded while the light travelled to us, which depends on what the Universe is made of. Astropy does these calculations for you.

Most of the galaxy evolution literature, including ZFOURGE, assumes a flat universe with $H_0 = 70$ km/s/Mpc and a matter density $\Omega_m = 0.3$. Always check which cosmology a paper uses before comparing your numbers to theirs.

In [ ]:
from astropy.cosmology import FlatLambdaCDM, Planck18

cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
print(cosmo)

There's also more than one kind of distance in an expanding universe, and each is used for something different:

| Method | Gives you | Used for |
|--------|-----------|----------|
| `cosmo.luminosity_distance(z)` | $d_L$ | Converting flux to luminosity |
| `cosmo.angular_diameter_distance(z)` | $d_A$ | Converting angular size to physical size |
| `cosmo.comoving_distance(z)` | $d_C$ | Volumes and number densities, clustering |
| `cosmo.distmod(z)` | $\mu = 5\log_{10}(d_L / 10\,\mathrm{pc})$ | Converting apparent to absolute magnitude |
| `cosmo.age(z)` | Age of the Universe at $z$ | |
| `cosmo.lookback_time(z)` | How long ago the light left | |
| `cosmo.kpc_proper_per_arcmin(z)` | Physical size of 1 arcminute at $z$ | Galaxy sizes |

All of them accept arrays and return Astropy Quantities.

In [ ]:
z = 2.0

print(f"Luminosity distance:       {cosmo.luminosity_distance(z):.0f}")
print(f"Angular diameter distance: {cosmo.angular_diameter_distance(z):.0f}")
print(f"Comoving distance:         {cosmo.comoving_distance(z):.0f}")
print(f"Age of the Universe at z = 2: {cosmo.age(z):.2f}")
print(f"Lookback time:                {cosmo.lookback_time(z):.2f}")
print(f"Age of the Universe today:    {cosmo.age(0):.2f}")

At $z = 2$ the light left its galaxy more than 10 billion years ago, when the Universe was less than a quarter of its current age. This is the era when galaxies were forming stars fastest.

Notice that the three distances are very different at high redshift. Using the wrong one is a classic mistake.

`Planck18` is a ready-made cosmology using the measurements from the Planck satellite. The differences from our simple model are small, but they matter for precision work:

In [ ]:
print(f"Our cosmology: d_L(z=2) = {cosmo.luminosity_distance(2):.0f}")
print(f"Planck 2018:   d_L(z=2) = {Planck18.luminosity_distance(2):.0f}")

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Looking Back in Time
  </h1>
</div>

1. Make an array `z_grid = np.array([0.1, 0.5, 1, 2, 3, 6])`. Using `cosmo`, print a table showing, for each redshift: the lookback time and the age of the Universe (in Gyr), the luminosity distance (in Gpc), and the physical size of one arcsecond (in kpc). *Hint: `cosmo.kpc_proper_per_arcmin(z)` gives kpc per arcminute, and there are 60 arcseconds in an arcminute.*
2. A typical ground-based image has a resolution of about 0.6 arcseconds. What physical size does this correspond to at $z = 2$? Store it, in kpc, as a Quantity called `res_z2`. How does it compare to the Milky Way's disc, which is about 30 kpc across?
3. Compare Hubble's law ($d = cz/H_0$) with the proper luminosity distance at $z = 0.023$, $0.5$ and $2$. At what redshift does Hubble's law stop being a good approximation?

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert abs(res_z2.to(u.kpc).value - 5.0) < 0.2, "res_z2 should be about 5 kpc. Check your unit conversions."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
z_grid = np.array([0.1, 0.5, 1, 2, 3, 6])

lookback = cosmo.lookback_time(z_grid)
age = cosmo.age(z_grid)
d_l = cosmo.luminosity_distance(z_grid).to(u.Gpc)
scale = cosmo.kpc_proper_per_arcmin(z_grid).to(u.kpc / u.arcsec)

print(f"{'z':>4} {'lookback':>9} {'age':>6} {'d_L':>7} {'kpc/arcsec':>11}")
for i in range(z_grid.size):
    print(f"{z_grid[i]:4.1f} {lookback[i].value:9.2f} {age[i].value:6.2f} {d_l[i].value:7.2f} {scale[i].value:11.2f}")

res_z2 = (0.6 * u.arcsec * cosmo.kpc_proper_per_arcmin(2)).to(u.kpc)
print(f"\n0.6 arcsec at z = 2 is {res_z2:.1f}, a sizeable fraction of a Milky Way-sized disc")

print()
for z in [0.023, 0.5, 2]:
    d_hubble = (const.c * z / cosmo.H0).to(u.Mpc)
    d_proper = cosmo.luminosity_distance(z)
    print(f"z = {z}: Hubble's law {d_hubble:.0f}, luminosity distance {d_proper:.0f} ({d_proper / d_hubble:.2f} times larger)")
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

At $z = 0.023$ the two agree to within about 1%, but by $z = 0.5$ Hubble's law is out by over 30%, and by $z = 2$ it's wrong by nearly a factor of 2. For anything beyond the very local Universe, use Astropy.

---
## 9. Bringing It Together

Let's go back to the eight galaxies from the end of Part 1, but this time store them as arrays. As before, the values are made up for practice but realistic for ZFOURGE.

In [ ]:
ids = np.array(["ZF-CDFS-1023", "ZF-CDFS-2210", "ZF-CDFS-3378", "ZF-COSMOS-0412",
                "ZF-COSMOS-1187", "ZF-COSMOS-2944", "ZF-UDS-0731", "ZF-UDS-1650"])
z = np.array([0.62, 0.91, 1.34, 0.75, 1.12, 0.58, 1.48, 0.83])
ks_flux = np.array([45.2, 12.8, 8.6, 60.1, 5.3, 22.4, 15.7, 9.9])     # µJy
u_v = np.array([1.85, 1.05, 1.70, 1.95, 0.85, 1.45, 1.62, 1.20])
v_j = np.array([1.10, 0.80, 1.75, 1.25, 0.55, 1.20, 0.95, 1.45])

Remember the UVJ criteria for quiescent galaxies from Part 1:

$$U-V > 1.3, \qquad V-J < 1.6, \qquad U-V > 0.88\,(V-J) + 0.59$$

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: A Mini Catalogue, Vectorised
  </h1>
</div>

Without using any loops:

1. Calculate the AB magnitude of every galaxy in the $K_s$ band, and store it in `ks_mag`.
2. Make a boolean mask called `quiescent` using the UVJ criteria.
3. Calculate each galaxy's **absolute** $K_s$ magnitude, $M = m - \mu(z)$, using `cosmo.distmod()`. Store it in `abs_ks`. *Hint: `distmod` returns a Quantity in magnitudes, so use `.value` to get plain numbers.*
4. Print the fraction of galaxies that are quiescent, and the mean absolute magnitude of each type.
5. Which galaxy is intrinsically brightest? Use `np.argmin` (remember, smaller magnitudes are brighter) to find its ID.

This ignores the **k-correction**, which accounts for the observed $K_s$ band sampling a different part of the spectrum at each redshift. You'd need it for real science, but it's fine for now.

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert np.sum(quiescent) == 3, "There should be 3 quiescent galaxies. Remember to use & and brackets."
assert abs(abs_ks[0] - (-23.11)) < 0.05, "abs_ks doesn't look right. Did you use cosmo.distmod(z).value?"
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
ks_mag = ab_mag(ks_flux)
quiescent = (u_v > 1.3) & (v_j < 1.6) & (u_v > 0.88 * v_j + 0.59)
abs_ks = ks_mag - cosmo.distmod(z).value

print(f"Quiescent fraction: {np.mean(quiescent):.0%}")
print(f"Mean absolute Ks, quiescent:    {np.mean(abs_ks[quiescent]):.2f}")
print(f"Mean absolute Ks, star-forming: {np.mean(abs_ks[~quiescent]):.2f}")

brightest = np.argmin(abs_ks)
print(f"Intrinsically brightest: {ids[brightest]} (M = {abs_ks[brightest]:.2f}, z = {z[brightest]})")
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

Here's a preview of what's coming in Part 4: a UVJ diagram of these galaxies. Read through the code to see how much of it you already recognise.

In [ ]:
quiescent = (u_v > 1.3) & (v_j < 1.6) & (u_v > 0.88 * v_j + 0.59)     # as in the exercise, in case you skipped it
vj_line = np.linspace(0.81, 1.6, 10)    # from where the sloped boundary meets U-V = 1.3, up to V-J = 1.6

plt.figure(figsize=(5, 5))
plt.scatter(v_j[~quiescent], u_v[~quiescent], color="tab:blue", label="Star-forming")
plt.scatter(v_j[quiescent], u_v[quiescent], color="tab:red", label="Quiescent")
plt.plot([0, 0.81], [1.3, 1.3], color="k")                     # horizontal part of the boundary
plt.plot(vj_line, 0.88 * vj_line + 0.59, color="k")            # sloped part
plt.plot([1.6, 1.6], [0.88 * 1.6 + 0.59, 2.5], color="k")      # vertical part
plt.xlim(0, 2)
plt.ylim(0.5, 2.5)
plt.xlabel("V - J")
plt.ylabel("U - V")
plt.legend()
plt.show()

<div class="practical-end" style="
  background: #f5f5f5;
  border-radius: 12px;
  padding: 20px 24px;
  text-align: center;
  border: 1px solid #ddd;
  margin: 2rem 0;
">
  <p style="font-size: 1.5rem; margin: 0 0 8px; text-align: center;">🎉</p>
  <h1 style="margin: 0 0 4px; color: #111; text-align: center;">End of Part 2</h1>
  <p style="margin: 0; color: #555; font-size: 0.9rem; text-align: center;">
    Well done! You've finished Part 2.
  </p>
</div>

<style>
@media (prefers-color-scheme: dark) {
  .practical-end { background: #1e1e1e !important; border-color: #333 !important; }
  .practical-end h1 { color: #f0f0f0 !important; }
  .practical-end p  { color: #999     !important; }
}
</style>

## What's Next?

In **Part 3** you'll work with real survey data: reading catalogues into tables, opening FITS files, downloading data from online archives, cleaning it up and matching objects between catalogues.

<a href="https://colab.research.google.com/github/mjcowley/Python_Intro/blob/main/03-Catalogues-and-Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Want more practice?

- [NumPy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [Astropy units and quantities](https://docs.astropy.org/en/stable/units/)
- [Astropy cosmology](https://docs.astropy.org/en/stable/cosmology/)
- [Learn Astropy tutorials](https://learn.astropy.org/)